In [2]:
from pathlib import Path
import sys

proj_root = Path("/Users/jerry/coding/rrs-SDP-pigments")
sys.path.insert(0, str(proj_root))

In [3]:
import numpy as np
import pandas as pd

from utils.seabass_loader import load_hplc_data, extract_pigment_columns

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
hplc_2024_11_30_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/940f89322d_PVST-SOPACE_KM2419-HPLC_20241130_R1.sb"
hplc_2025_10_14_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/10d10ff7e5_PVST_SOPACE_TN444-HPLC_20251014.sb"
hplc_2025_11_03_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/32c3f67fb5_PVST_SOPACE_TN440-HPLC_20251103.sb"

In [6]:
hplc_2024_11_30 = load_hplc_data(hplc_2024_11_30_path)
hplc_2025_10_14 = load_hplc_data(hplc_2025_10_14_path)
hplc_2025_11_03 = load_hplc_data(hplc_2025_11_03_path)

In [7]:
shapes = pd.Series({
    "hplc_2024_11_30": hplc_2024_11_30.shape,
    "hplc_2025_10_14": hplc_2025_10_14.shape,
    "hplc_2025_11_03": hplc_2025_11_03.shape,
})
shapes

hplc_2024_11_30    (49, 53)
hplc_2025_10_14    (68, 53)
hplc_2025_11_03    (13, 53)
dtype: object

In [8]:
def lon_lat_ranges(df):
    lon = pd.to_numeric(df['lon'])
    lat = pd.to_numeric(df['lat'])
    return pd.Series({
        'lon_min': lon.min(),
        'lon_max': lon.max(),
        'lat_min': lat.min(),
        'lat_max': lat.max(),
    })

ranges = pd.DataFrame({
    'hplc_2024_11_30': lon_lat_ranges(hplc_2024_11_30),
    'hplc_2025_10_14': lon_lat_ranges(hplc_2025_10_14),
    'hplc_2025_11_03': lon_lat_ranges(hplc_2025_11_03),
}).T
ranges


,lon_min,lon_max,lat_min,lat_max
hplc_2024_11_30,-157.69040,-132.3535,-14.2475,20.65700
hplc_2025_10_14,85.99957,88.5087,9.9134,13.94745
hplc_2025_11_03,148.20800,149.0300,7.6510,15.00000


In [9]:
def date_ranges(df):
    dates = pd.to_datetime(df['date'].astype('string'))
    return dates

date_ranges(hplc_2025_10_14)

0    2025-05-06
1    2025-05-06
2    2025-05-06
3    2025-05-09
4    2025-05-09
        ...    
63   2025-06-14
64   2025-06-14
65   2025-06-15
66   2025-06-15
67   2025-06-15
Name: date, Length: 68, dtype: datetime64[ns]

### 2025-10-14, Bay of Bengal

In [10]:
import rskit as rs
from rskit.plugins import NasaEarthdata
nasa = NasaEarthdata()
rs.plugins.get_params_schema('nasa_earthdata')

{'required_fields': ['collection_concept_id'],
 'optional_fields': ['cloud_cover',
  'sort_key',
  'max_granules',
  'variables',
  'drop_nan_lines'],
 'field_descriptions': {'collection_concept_id': 'CMR collection concept ID.',
  'cloud_cover': 'Tuple of (min_percent, max_percent) for cloud cover filtering.',
  'sort_key': "CMR sort key (e.g., '-start_date').",
  'max_granules': 'Maximum number of granules to return/download.',
  'variables': 'List of variable names to subset during client-side processing (Harmony ignores variable subsetting).',
  'drop_nan_lines': 'Drop all-NaN lines during client-side subsetting.'},
 'notes': ['Query.time(...) and Query.region(...) are required for NASA Earthdata downloads.']}

In [11]:
hplc_2024_11_30_bbox = (-157.6904, -14.2475, -132.3535, 20.657)
hplc_2025_10_14_bbox = (85.9995703, 9.9134, 88.5087, 13.94745)
hplc_2025_11_03_bbox = (148.208, 7.651, 149.03, 15.0)

collection_id = nasa.get_collection_concept_id(short_name='PACE_OCI_L2_AOP', version='3.1')

query = (
    rs.query(source='nasa_earthdata')
    .region(bbox=hplc_2025_10_14_bbox)
    .time(start='2025-05-06', end='2025-06-15')
    .with_params(collection_concept_id=collection_id)
)
nasa.download_subsetted_data(
    query,
    destination=Path('/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs'),
    mask_out_of_bounds=True
)

422 Client Error: Unprocessable Entity for url: https://harmony.earthdata.nasa.gov/C3620139598-OB_CLOUD/ogc-api-coverages/1.0.0/collections/all/coverage/rangeset?subset=lon%2885.9995703%3A88.5087%29&subset=lat%289.9134%3A13.94745%29&subset=time%28%222025-05-06T00%3A00%3A00Z%22%3A%222025-06-15T00%3A00%3A00Z%22%29


/Users/jerry/coding/RS-Kit/src/rskit/plugins/nasa_earthdata/base.py:297: UserWarning: Harmony subsetting failed, attemping to subset locally.
  warnings.warn("Harmony subsetting failed, attemping to subset locally.")


Skipping /Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250519T060350.L2.OC_AOP.V3_1.nc: there is either no spatial or no temporal overlap detected
Skipping /Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250608T075939.L2.OC_AOP.V3_1.nc: there is either no spatial or no temporal overlap detected


['/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250506T063633.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250507T071152.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250510T071924.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250511T061620.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250511T075441.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250512T065140.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250513T072658.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_

In [12]:
sst_id = 'C1615905770-OB_DAAC'
sss_id = 'C2208422957-POCLOUD'

In [11]:
hplc = extract_pigment_columns(hplc_2025_10_14)
hplc.notna().mean()

station      1.000000
date         1.000000
time         1.000000
lon          1.000000
lat          1.000000
Allo         0.000000
But-fuco     0.941176
Chl_c1c2     1.000000
Chl_c3       1.000000
DV_Chl_a     1.000000
Fuco         0.941176
Hex-fuco     1.000000
MV_Chl_b     1.000000
Neo          0.147059
Perid        1.000000
Tchl         1.000000
Tot_Chl_a    1.000000
Viola        0.588235
Zea          1.000000
dtype: float64

In [18]:
import xarray as xr
path = Path('/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/sss/SMAP_L3_SSS_20250502_8DAYS_V5.0.nc')

with xr.open_dataset(path) as ds:
    print(ds)

<xarray.Dataset> Size: 37MB
Dimensions:               (latitude: 720, longitude: 1440, time: 1)
Coordinates:
  * latitude              (latitude) float32 3kB 89.88 89.62 ... -89.62 -89.88
  * longitude             (longitude) float32 6kB -179.9 -179.6 ... 179.6 179.9
  * time                  (time) datetime64[ns] 8B 2025-05-02T12:00:00
Data variables:
    smap_sss              (latitude, longitude) float32 4MB ...
    anc_sss               (latitude, longitude) float32 4MB ...
    anc_sst               (latitude, longitude) float32 4MB ...
    smap_spd              (latitude, longitude) float32 4MB ...
    smap_high_spd         (latitude, longitude) float32 4MB ...
    weight                (latitude, longitude) float32 4MB ...
    land_fraction         (latitude, longitude) float32 4MB ...
    ice_fraction          (latitude, longitude) float32 4MB ...
    smap_sss_uncertainty  (latitude, longitude) float32 4MB ...
Attributes: (12/40)
    title:                       SMAP 0.25x0.25 d

### Remaining items before end-to-end PVST run

**Fixed:**
- ~~Wavelength extraction bug~~ — `_get_l2_rrs_wavelengths` now tries `wavelength_3d` (172 Rrs bands) before `wavelength` (286 total sensor bands), with per-candidate error handling in the `sensor_band_parameters` fallback.
- ~~SST/SSS `open_mfdataset` failure~~ — Replaced with per-matchup file selection using `time_coverage_start/end` attributes (handles SMAP day-of-year timestamps). No dask dependency required.

**Still open:**
1. **SST product: daily vs 8-day** — Current files are 8-day composites (`L3m.8D`). The Kramer et al. methodology uses daily SST. Decide whether this matters for your validation or if 8-day composites are acceptable.
2. **Smoothing placement** — Smoothing currently happens on raw Rrs at L2 extraction time (`pace_l2.py`). The written spec says smoothing applies to Rrs before the model sees it (which is what the code does), but verify this matches the original Kramer et al. pipeline.
3. **Evaluation outputs** — `sdp_results.nc` currently writes predictions + matchup metadata but does NOT embed in-situ HPLC pigment values. You still need: HPLC column name → model pigment name mapping, parity plots, Bland–Altman plots, per-pigment RMSE/bias.
4. **Debug print in prediction.py** — `run_sdp` prints the full derivative matrix; remove once debugging is done.